# DataFog PII-NER v1.1 — Full Training

Train the complete model on all 360K+ examples from AI4Privacy, Nemotron-PII, and Gretel datasets.

**v1.1 improvements over v1:**
- Tier-weighted CRF loss (3x weight for Tier 1 entities like SSN, Credit Card)
- Oversampling of rare entity examples (Tier 1 x3)
- Halved backbone LR (1e-5) with cosine schedule to prevent late-epoch regression
- 15 epochs (from 10) with 500-step warmup

**Compute requirements:**
- GPU: A100 (40GB) recommended, T4 (16GB) possible with smaller batch size
- Training time: ~6-8 hours on A100, ~18-24 hours on T4
- Disk: ~5GB for datasets + checkpoints

**Instructions:**
1. Runtime → Change runtime type → **A100 GPU** (or T4)
2. Set your WandB API key in the cell below
3. Click **Run All**

## 1. Setup

In [ ]:
import os, sys

# Clone or force-update to latest
if not os.path.exists("/content/datafog-labs"):
    !git clone https://github.com/DataFog/datafog-labs.git /content/datafog-labs
else:
    !cd /content/datafog-labs && git fetch origin && git reset --hard origin/main

# Force reinstall to ensure latest code (pip may cache stale editable installs)
!pip install -e "/content/datafog-labs/pii-ner-v1[dev]" --force-reinstall --no-deps -q
!pip install -e "/content/datafog-labs/pii-ner-v1[dev]" -q  # install deps if missing

sys.path.insert(0, "/content/datafog-labs/pii-ner-v1/src")

# Reload to pick up fresh install
import importlib
import datafog_pii_ner
importlib.reload(datafog_pii_ner)
print(f"datafog_pii_ner loaded from: {datafog_pii_ner.__file__}")
!cd /content/datafog-labs && git log -1 --oneline

# Verify critical fixes are present
import inspect
from datafog_pii_ner.training.train import PiiTrainer
source = inspect.getsource(PiiTrainer.create_optimizer)
assert "eps" in source, "MISSING: eps=1.0 fix for AdamW NaN"
assert "model.float()" in source or "self.model.float()" in source, "MISSING: FP32 master weights fix"
print("Verified: eps=1.0 and FP32 master weights fixes are present")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    try:
        props = torch.cuda.get_device_properties(0)
        mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
        print(f"Memory: {mem / 1e9:.1f} GB")
    except Exception:
        print("Memory: (could not detect)")
else:
    raise RuntimeError("No GPU found — full training requires a GPU")

In [ ]:
# WandB login
import wandb
wandb.login()  # Will prompt for API key if not set

## 2. Configuration

In [ ]:
# Training configuration — v1.1
# Adjust batch sizes based on your GPU memory:
#   A100 (40GB): batch_size=32, grad_accum=1 -> effective=32
#   T4 (16GB):   batch_size=8,  grad_accum=4 -> effective=32

CONFIG = {
    # Model
    "backbone": "microsoft/deberta-v3-xsmall",
    "max_seq_len": 256,
    "max_char_len": 20,
    "dropout": 0.1,
    
    # Training — v1.1 updates
    "epochs": 15,                         # was 10 in v1
    "batch_size": 32,
    "gradient_accumulation_steps": 1,
    "lr_backbone": 1e-5,                  # halved from 2e-5 to prevent late-epoch regression
    "lr_head": 1e-3,
    "lr_scheduler_type": "cosine",        # smoother decay than linear
    "warmup_steps": 500,                  # ~1 epoch warmup
    "weight_decay": 0.01,
    
    # Tier-weighted CRF loss: sequences with critical PII get higher loss weight
    "tier_weights": {1: 3.0, 2: 2.0, 3: 1.5, 4: 1.0},
    
    # Oversampling: duplicate examples containing Tier 1 entities
    "oversample_tiers": [1],
    "oversample_factor": 3,
    
    # Mixed precision strategy (set by preflight check):
    # - BF16 is preferred on A100/H100: no gradient scaler, same exponent range as FP32
    # - FP32 is the guaranteed fallback: slower but always correct
    "fp16": False,
    "bf16": False,  # Set by preflight check below
    
    # Data
    "val_ratio": 0.1,
    "test_ratio": 0.1,
    "seed": 42,
    
    # Output
    "output_dir": "/content/pii_ner_v1.1_output",
    "run_name": "pii-ner-v1.1-a100",
}

# Auto-adjust batch size for GPU memory
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_properties(0).major
    if gpu_mem < 20:
        print(f"GPU: {gpu_name} ({gpu_mem:.0f}GB) — adjusting batch size")
        CONFIG["batch_size"] = 8
        CONFIG["gradient_accumulation_steps"] = 4
    else:
        print(f"GPU: {gpu_name} ({gpu_mem:.0f}GB) — using batch_size=32")
    
    # BF16 requires compute capability >= 8.0 (A100, H100, etc.)
    if cc >= 8:
        print(f"Compute capability {cc}.x — BF16 available (preflight will verify)")
        CONFIG["_try_bf16"] = True
    else:
        print(f"Compute capability {cc}.x — BF16 not available, using FP32")
        CONFIG["_try_bf16"] = False

effective_batch = CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"]
print(f"Effective batch size: {effective_batch}")
print(f"Backbone LR: {CONFIG['lr_backbone']} (halved from v1)")
print(f"Scheduler: {CONFIG['lr_scheduler_type']}")
print(f"Tier weights: {CONFIG['tier_weights']}")
print(f"Oversampling: Tier {CONFIG['oversample_tiers']} x{CONFIG['oversample_factor']}")

## 2.5 Preflight Check (auto-selects precision)

Tries BF16 first (faster, no gradient scaler). If NaN detected, falls back to FP32 (slower but guaranteed correct). FP16 is **not** attempted — incompatible with custom optimizer param groups on Accelerate 1.x+.

In [ ]:
import math
import gc
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, TrainingArguments
from datafog_pii_ner.data.collator import PiiDataCollator
from datafog_pii_ner.data.label_schema import NUM_LABELS
from datafog_pii_ner.model.pii_model import PiiNerConfig, PiiNerModel
from datafog_pii_ner.training.train import PiiTrainer


def run_preflight(batch_size, seq_len, max_char_len, backbone, lr_backbone, lr_head, fp16, bf16):
    """Run 5 training steps and return (passed: bool, loss: float)."""
    label = "BF16" if bf16 else "FP16" if fp16 else "FP32"
    print(f"\n--- Testing {label} (batch_size={batch_size}) ---")
    
    tokenizer = AutoTokenizer.from_pretrained(backbone)
    n_samples = batch_size * 2

    fake_data = {
        "input_ids": np.random.randint(1, 1000, (n_samples, seq_len)).tolist(),
        "attention_mask": np.ones((n_samples, seq_len), dtype=int).tolist(),
        "labels": np.random.randint(0, NUM_LABELS, (n_samples, seq_len)).tolist(),
        "char_ids": np.random.randint(0, 100, (n_samples, seq_len, max_char_len)).tolist(),
    }
    ds = Dataset.from_dict(fake_data)
    collator = PiiDataCollator(tokenizer=tokenizer, max_char_len=max_char_len)
    config = PiiNerConfig(backbone=backbone, num_labels=NUM_LABELS)
    model = PiiNerModel(config)

    args = TrainingArguments(
        output_dir="/tmp/preflight_check",
        num_train_epochs=1,
        per_device_train_batch_size=batch_size,
        learning_rate=lr_backbone,
        fp16=fp16,
        bf16=bf16,
        report_to="none",
        logging_steps=1,
        max_steps=5,
        remove_unused_columns=False,
        save_strategy="no",
    )

    trainer = PiiTrainer(
        model=model, args=args, train_dataset=ds, data_collator=collator,
        lr_backbone=lr_backbone, lr_head=lr_head,
    )

    try:
        result = trainer.train()
        loss = result.training_loss
    except Exception as e:
        print(f"  FAIL: {type(e).__name__}: {e}")
        del model, trainer, ds, collator, args
        gc.collect(); torch.cuda.empty_cache()
        return False, float("nan")

    # NaN checks
    passed = True

    if math.isnan(loss) or math.isinf(loss):
        print(f"  FAIL: Training loss is {loss}")
        passed = False
    else:
        print(f"  PASS: Training loss = {loss:.4f}")

    nan_params = [n for n, p in model.named_parameters() if torch.isnan(p).any()]
    if nan_params:
        print(f"  FAIL: NaN in {len(nan_params)} parameter tensors")
        passed = False
    else:
        print(f"  PASS: All parameter tensors are finite")

    log_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    if log_losses and any(math.isnan(l) for l in log_losses):
        print(f"  FAIL: NaN in step losses: {log_losses}")
        passed = False
    elif len(log_losses) >= 2 and log_losses[-1] < log_losses[0]:
        print(f"  PASS: Loss decreasing ({log_losses[0]:.1f} → {log_losses[-1]:.1f})")

    del model, trainer, ds, collator, args, result
    gc.collect(); torch.cuda.empty_cache()
    return passed, loss


# === Run preflight with auto-fallback ===
print("=== PREFLIGHT CHECK ===")

preflight_ok = False
preflight_args = dict(
    batch_size=CONFIG["batch_size"],
    seq_len=CONFIG["max_seq_len"],
    max_char_len=CONFIG["max_char_len"],
    backbone=CONFIG["backbone"],
    lr_backbone=CONFIG["lr_backbone"],
    lr_head=CONFIG["lr_head"],
)

# Try 1: BF16 (if GPU supports it)
if CONFIG.get("_try_bf16", False):
    ok, loss = run_preflight(**preflight_args, fp16=False, bf16=True)
    if ok:
        CONFIG["fp16"] = False
        CONFIG["bf16"] = True
        preflight_ok = True
        print(f"\n=== BF16 PREFLIGHT PASSED (loss={loss:.4f}) ===")
    else:
        print(f"\nBF16 failed — falling back to FP32")

# Try 2: FP32 (guaranteed fallback)
if not preflight_ok:
    ok, loss = run_preflight(**preflight_args, fp16=False, bf16=False)
    if ok:
        CONFIG["fp16"] = False
        CONFIG["bf16"] = False
        preflight_ok = True
        print(f"\n=== FP32 PREFLIGHT PASSED (loss={loss:.4f}) ===")
    else:
        raise RuntimeError(
            "PREFLIGHT FAILED on FP32 — something is fundamentally broken. "
            "Check model architecture and data pipeline."
        )

precision = "BF16" if CONFIG["bf16"] else "FP16" if CONFIG["fp16"] else "FP32"
print(f"\nSelected precision: {precision}")
print(f"Training will proceed with: fp16={CONFIG['fp16']}, bf16={CONFIG['bf16']}")

## 3. Load Data (all 360K+ examples)

In [ ]:
from transformers import AutoTokenizer
from datafog_pii_ner.data.dataset import load_pii_datasets
from datafog_pii_ner.data.label_schema import NUM_LABELS

tokenizer = AutoTokenizer.from_pretrained(CONFIG["backbone"])

print("Loading all datasets (this may take a few minutes)...")
datasets = load_pii_datasets(
    tokenizer=tokenizer,
    max_seq_len=CONFIG["max_seq_len"],
    max_char_len=CONFIG["max_char_len"],
    val_ratio=CONFIG["val_ratio"],
    test_ratio=CONFIG["test_ratio"],
    seed=CONFIG["seed"],
    oversample_tiers=CONFIG.get("oversample_tiers"),
    oversample_factor=CONFIG.get("oversample_factor", 2),
)

print(f"\nDataset sizes:")
print(f"  Train:      {len(datasets['train']):,}")
print(f"  Validation: {len(datasets['validation']):,}")
print(f"  Test:       {len(datasets['test']):,}")
print(f"  Total:      {sum(len(datasets[s]) for s in datasets):,}")
print(f"  Labels:     {NUM_LABELS}")
if CONFIG.get("oversample_tiers"):
    print(f"  Oversampling: Tier {CONFIG['oversample_tiers']} x{CONFIG['oversample_factor']} (applied to train only)")

## 4. Initialize Model

In [ ]:
from datafog_pii_ner.model.pii_model import PiiNerConfig, PiiNerModel
from datafog_pii_ner.data.label_schema import build_label_weights

config = PiiNerConfig(
    backbone=CONFIG["backbone"],
    num_labels=NUM_LABELS,
    dropout=CONFIG["dropout"],
)
model = PiiNerModel(config)

# Set tier-weighted CRF loss
tier_weights_cfg = CONFIG.get("tier_weights")
if tier_weights_cfg:
    weights = build_label_weights(tier_weights_cfg)
    model.crf_head.set_label_weights(torch.tensor(weights, dtype=torch.float32))
    print(f"Tier-weighted CRF loss enabled: {tier_weights_cfg}")

param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {param_count:,}")
print(f"Trainable parameters: {trainable_count:,}")

## 5. Train

In [ ]:
from transformers import TrainingArguments
from datafog_pii_ner.data.collator import PiiDataCollator
from datafog_pii_ner.training.metrics import compute_metrics
from datafog_pii_ner.training.train import PiiTrainer

collator = PiiDataCollator(tokenizer=tokenizer, max_char_len=CONFIG["max_char_len"])

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["lr_backbone"],
    lr_scheduler_type=CONFIG.get("lr_scheduler_type", "linear"),
    warmup_steps=CONFIG.get("warmup_steps", 0),
    weight_decay=CONFIG["weight_decay"],
    fp16=CONFIG.get("fp16", False),
    bf16=CONFIG.get("bf16", False),
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="overall_f1",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name=CONFIG["run_name"],
    logging_steps=50,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    save_total_limit=3,
)

print(f"Backbone LR: {CONFIG['lr_backbone']}, Head LR: {CONFIG['lr_head']}")
print(f"Scheduler: {CONFIG.get('lr_scheduler_type', 'linear')}, Warmup steps: {CONFIG.get('warmup_steps', 0)}")
print(f"Mixed precision: {'bf16' if CONFIG.get('bf16') else 'fp16' if CONFIG.get('fp16') else 'none'}")

# PiiTrainer handles differential learning rates internally via create_optimizer().
# The backbone param group uses eps=1.0 to dampen AdamW's adaptive scaling,
# preventing NaN weight updates caused by bias-correction amplification at step 1.
trainer = PiiTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    lr_backbone=CONFIG["lr_backbone"],
    lr_head=CONFIG["lr_head"],
)

print(f"\nStarting training for {CONFIG['epochs']} epochs...")
train_result = trainer.train()
print(f"\nTraining complete. Final loss: {train_result.training_loss:.4f}")

## 6. Evaluate on Test Set

In [ ]:
print("Evaluating on held-out test set...")
test_results = trainer.evaluate(datasets["test"])

print("=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"  Overall F1:        {test_results.get('eval_overall_f1', 0):.4f}")
print(f"  Overall Precision: {test_results.get('eval_overall_precision', 0):.4f}")
print(f"  Overall Recall:    {test_results.get('eval_overall_recall', 0):.4f}")
print()

# Tier recalls
for tier in [1, 2, 3, 4]:
    key = f"eval_tier_{tier}_recall"
    if key in test_results:
        target = {1: 0.98, 2: 0.95, 3: 0.90, 4: 0.85}[tier]
        actual = test_results[key]
        status = "PASS" if actual >= target else "FAIL"
        print(f"  Tier {tier} recall: {actual:.4f}  (target >= {target})  [{status}]")

# Per-type F1 (top 20)
print("\nPer-entity F1 (top 20):")
type_metrics = [(k, v) for k, v in test_results.items() if k.startswith("eval_type_") and k.endswith("_f1")]
type_metrics.sort(key=lambda x: x[1], reverse=True)
for k, v in type_metrics[:20]:
    name = k.replace("eval_type_", "").replace("_f1", "")
    print(f"  {name:30s} {v:.4f}")

## 7. Save Model

In [ ]:
save_path = f"{CONFIG['output_dir']}/best_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# Also save the config for reproducibility
import json
with open(f"{save_path}/training_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
print(f"Training config saved to {save_path}/training_config.json")

# Save test results
with open(f"{save_path}/test_results.json", "w") as f:
    json.dump({k: float(v) if hasattr(v, '__float__') else v for k, v in test_results.items()}, f, indent=2)
print(f"Test results saved to {save_path}/test_results.json")

# Save as v1.1
print(f"\nModel version: v1.1")
print(f"Key improvements: tier-weighted CRF loss, oversampling, cosine LR schedule")

## 8. Download Model (optional)

Run this cell to download the trained model from Colab.

In [ ]:
# Zip and download the model
import shutil
shutil.make_archive("/content/pii-ner-v1-model", "zip", save_path)
print(f"Model archived to /content/pii-ner-v1-model.zip")

try:
    from google.colab import files
    files.download("/content/pii-ner-v1-model.zip")
except ImportError:
    print("Not running in Colab — download manually")